# Projeto de Análise Exploratória de Dados (EDA)
## Planejamento — Dataset, Problema e Perguntas Norteadoras

> Este notebook registra a **primeira versão do projeto de EDA** que será desenvolvido no Módulo 4.  
> Aqui não realizamos a análise em si, mas sim o planejamento que vai guiá-la: a escolha do dataset, a definição do problema de negócio e as perguntas que queremos responder com os dados.

---

Um dos erros mais comuns de quem está começando em Ciência de Dados é **partir direto para os gráficos** sem saber o que está procurando.  
A análise exploratória eficiente começa antes do código — ela começa com **perguntas bem formuladas**.

> *"Se eu tivesse uma hora para resolver um problema, passaria 55 minutos pensando nele e 5 minutos pensando em soluções."*  
> — Atribuído a Albert Einstein

---
## 1. O Dataset: Olist Brazilian E-commerce

### 1.1 Origem e contexto

O dataset utilizado neste projeto é o **[Olist Brazilian E-commerce Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)**, disponibilizado publicamente no Kaggle pela própria empresa.

A **Olist** é uma plataforma de marketplace que conecta pequenos lojistas a grandes canais de venda no Brasil. Em vez de cada lojista criar sua própria loja, eles utilizam a infraestrutura da Olist para vender em múltiplos marketplaces simultaneamente.

Os dados cobrem pedidos realizados **entre setembro de 2016 e outubro de 2018**, com informações sobre:
- status e datas dos pedidos
- localização dos clientes e vendedores
- categorias de produtos
- formas e valores de pagamento
- avaliações e comentários dos clientes

### 1.2 Estrutura do dataset

O dataset é composto por **9 tabelas relacionadas**, conectadas por chaves como `order_id`, `customer_id` e `product_id`. Para o nosso projeto de EDA, trabalharemos principalmente com as seguintes tabelas:

| Arquivo | Descrição | Linhas (aprox.) |
|---|---|---|
| `olist_orders_dataset.csv` | Pedidos, status e datas de entrega | ~100 mil |
| `olist_order_reviews_dataset.csv` | Avaliações e notas dos clientes | ~100 mil |
| `olist_order_payments_dataset.csv` | Formas e valores de pagamento | ~104 mil |
| `olist_customers_dataset.csv` | Localização dos clientes | ~100 mil |
| `olist_order_items_dataset.csv` | Itens e preços de cada pedido | ~113 mil |
| `olist_products_dataset.csv` | Categorias e características dos produtos | ~33 mil |

### 1.3 Por que esse dataset é ideal para uma EDA introdutória?

- **Contexto familiar:** todo mundo já fez uma compra online — o domínio é imediato
- **Dados reais e imperfeitos:** existem valores ausentes, outliers e decisões de limpeza a serem tomadas
- **Riqueza analítica:** permite explorar tempo, espaço, comportamento do consumidor e qualidade de serviço
- **Perguntas de negócio reais:** as análises têm impacto prático — não são exercícios artificiais
- **Escala adequada:** grande o suficiente para ser realista, pequeno o suficiente para rodar em qualquer máquina

---
## 2. Inspeção Inicial do Dataset

Antes de formular o problema com precisão, é fundamental ver o que os dados dizem. Esta seção carrega as tabelas principais e faz uma inspeção rápida para confirmar que temos o que precisamos.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)

# Caminho base dos datasets — ajuste se necessário
BASE = Path("../datasets")

print("Bibliotecas carregadas com sucesso.")

In [ ]:
# Carregando as tabelas principais
orders   = pd.read_csv(BASE / "olist_orders_dataset.csv")
reviews  = pd.read_csv(BASE / "olist_order_reviews_dataset.csv")
payments = pd.read_csv(BASE / "olist_order_payments_dataset.csv")
customers = pd.read_csv(BASE / "olist_customers_dataset.csv")
items    = pd.read_csv(BASE / "olist_order_items_dataset.csv")
products = pd.read_csv(BASE / "olist_products_dataset.csv")

print("Tabelas carregadas:")
tabelas = {
    "orders":    orders,
    "reviews":   reviews,
    "payments":  payments,
    "customers": customers,
    "items":     items,
    "products":  products
}
for nome, df in tabelas.items():
    print(f"  {nome:12s}: {df.shape[0]:>7,} linhas × {df.shape[1]:>2} colunas")

In [ ]:
# Visão rápida das tabelas mais importantes para o projeto
print("=== Primeiros registros: orders ===")
display(orders.head(3))

print("\n=== Primeiros registros: reviews ===")
display(reviews.head(3))

print("\n=== Primeiros registros: payments ===")
display(payments.head(3))

In [ ]:
# Verificação rápida de valores ausentes nas tabelas
print("Valores ausentes por tabela:")
for nome, df in tabelas.items():
    total_ausentes = df.isnull().sum().sum()
    if total_ausentes > 0:
        print(f"\n  [{nome}] — {total_ausentes} valores ausentes")
        print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

In [ ]:
# Distribuição de notas de avaliação — primeiro sinal do que vamos explorar
print("Distribuição das notas de avaliação (review_score):")
dist_notas = reviews["review_score"].value_counts().sort_index()
dist_notas_pct = (dist_notas / dist_notas.sum() * 100).round(1)

resumo = pd.DataFrame({"quantidade": dist_notas, "percentual_%": dist_notas_pct})
print(resumo)
print(f"\nNota média geral: {reviews['review_score'].mean():.2f}")

In [ ]:
# Distribuição dos status dos pedidos — entendendo o ciclo de vida
print("Distribuição dos status de pedidos:")
print(orders["order_status"].value_counts())
print(f"\nPercentual de pedidos entregues: {(orders['order_status'] == 'delivered').mean():.1%}")

---
## 3. O Problema de Negócio

### 3.1 Definição do problema

Com base na inspeção inicial dos dados, identificamos que o dataset da Olist oferece uma oportunidade de análise com impacto real:

---

###  Problema central

> **O que determina a satisfação do cliente em um pedido na Olist — e como o desempenho operacional de entrega influencia essa avaliação?**

---

### 3.2 Por que este problema é relevante?

A avaliação do cliente (`review_score`, de 1 a 5 estrelas) é um dos indicadores mais importantes de qualidade em qualquer plataforma de e-commerce. Ela afeta diretamente:

- a **reputação dos vendedores** dentro da plataforma
- a **taxa de recompra** dos clientes
- o **posicionamento dos produtos** nos resultados de busca
- a **percepção geral da Olist** como marketplace confiável

Entender o que faz um cliente dar 1 estrela vs. 5 estrelas é uma pergunta de negócio com valor prático imediato.

### 3.3 Hipóteses iniciais

Antes de analisar os dados, levantamos hipóteses baseadas no senso comum e em conhecimento de domínio:

| # | Hipótese | Raciocínio |
|---|---|---|
| H1 | Pedidos atrasados recebem notas mais baixas | Clientes frustrados com o prazo tendem a avaliar pior |
| H2 | Pedidos mais baratos têm notas mais altas | Expectativas menores são mais fáceis de superar |
| H3 | Categorias de produtos influenciam as notas | Produtos de beleza e moda podem ter mais frustração com tamanho/cor |
| H4 | A nota média varia por região do Brasil | Infraestrutura logística é diferente entre estados |
| H5 | Pedidos com mais itens tendem a ter mais problemas | Mais itens = maior chance de algum ter problema |

A EDA do Módulo 4 vai **confirmar, refutar ou refinar** cada uma dessas hipóteses.

---
## 4. Perguntas Norteadoras

As perguntas norteadoras são o coração do planejamento de uma EDA. Elas transformam o problema amplo em **questões específicas, mensuráveis e respondíveis com os dados disponíveis**.

Organizamos as perguntas em quatro eixos analíticos:

---

###  Eixo 1 — Comportamento dos Pedidos

> *Entender como os pedidos se distribuem no tempo e no espaço.*

**P1.1** — Como evoluiu o volume de pedidos ao longo do tempo? Existe uma tendência de crescimento clara?

**P1.2** — Existem padrões sazonais? Quais meses ou períodos concentram mais pedidos (ex: Black Friday, Natal)?

**P1.3** — Em qual dia da semana e horário do dia os clientes mais compram?

**P1.4** — Quais estados do Brasil concentram mais clientes? A demanda é geograficamente concentrada ou distribuída?

---

###  Eixo 2 — Desempenho Operacional (Entrega)

> *Avaliar a eficiência logística e identificar gargalos.*

**P2.1** — Qual é o prazo médio de entrega? Como ele se distribui?

**P2.2** — Qual percentual de pedidos é entregue antes da data estimada? E após?

**P2.3** — O prazo de entrega varia de acordo com a região de destino? Quais estados têm as entregas mais rápidas e mais lentas?

**P2.4** — O desempenho operacional melhorou ao longo do tempo, à medida que a Olist cresceu?

---

###  Eixo 3 — Satisfação do Cliente (Avaliações)

> *Investigar o que influencia a nota que o cliente dá ao pedido.*

**P3.1** — Qual é a distribuição das notas de avaliação? A maioria dos clientes está satisfeita?

**P3.2** — Pedidos atrasados recebem notas significativamente mais baixas do que pedidos entregues no prazo?

**P3.3** — Existe relação entre o prazo de entrega (em dias) e a nota recebida?

**P3.4** — A nota média varia por estado de destino? Regiões com pior logística têm clientes menos satisfeitos?

**P3.5** — Existe diferença na satisfação entre categorias de produtos?

---

###  Eixo 4 — Perfil Financeiro dos Pedidos

> *Entender o padrão de compra e pagamento dos clientes.*

**P4.1** — Qual é o valor médio dos pedidos? Como ele se distribui?

**P4.2** — Quais são as formas de pagamento mais usadas? O cartão de crédito domina?

**P4.3** — Pedidos pagos em mais parcelas têm alguma relação com a satisfação do cliente?

**P4.4** — Quais categorias de produtos geram maior valor médio por pedido?

---
## 5. Variáveis-Chave do Projeto

Para responder às perguntas acima, precisaremos criar um **DataFrame analítico consolidado**, unindo as principais tabelas do dataset.

### 5.1 Variável-alvo

| Variável | Tabela | Descrição |
|---|---|---|
| `review_score` | `olist_order_reviews_dataset.csv` | Nota de 1 a 5 dada pelo cliente ao pedido |

### 5.2 Variáveis explicativas planejadas

| Variável | Origem | Tipo | Uso esperado |
|---|---|---|---|
| `prazo_entrega_dias` | Calculada (orders) | Numérica | Eixo 2 e 3 |
| `atraso_dias` | Calculada (orders) | Numérica | Eixo 2 e 3 |
| `entregou_no_prazo` | Calculada (orders) | Booleana | Eixo 2 e 3 |
| `order_status` | orders | Categórica | Eixo 1 e 3 |
| `customer_state` | customers | Categórica | Eixo 1 e 3 |
| `payment_type` | payments | Categórica | Eixo 4 |
| `payment_value` | payments | Numérica | Eixo 4 |
| `payment_installments` | payments | Numérica | Eixo 4 |
| `product_category_name` | products | Categórica | Eixo 3 e 4 |
| `price` | items | Numérica | Eixo 4 |
| `ano_compra`, `mes_compra` | Calculadas (orders) | Temporal | Eixo 1 |

### 5.3 Prévia da construção do DataFrame analítico

In [ ]:
# Prévia: construção do DataFrame analítico que será usado na EDA completa
# Esta é a lógica de join que vamos aplicar no Módulo 4

# Passo 1: converter datas no orders
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in colunas_data:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Passo 2: criar indicadores de entrega
orders["prazo_entrega_dias"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["atraso_dias"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["entregou_no_prazo"] = orders["atraso_dias"] <= 0
orders["ano_compra"]  = orders["order_purchase_timestamp"].dt.year
orders["mes_compra"]  = orders["order_purchase_timestamp"].dt.month

# Passo 3: nota média por pedido (alguns pedidos têm mais de uma avaliação)
nota_por_pedido = (
    reviews
    .groupby("order_id")["review_score"]
    .mean()
    .reset_index()
    .rename(columns={"review_score": "nota_media"})
)

# Passo 4: valor total por pedido (pagamentos)
valor_por_pedido = (
    payments
    .groupby("order_id")
    .agg(
        valor_total=("payment_value", "sum"),
        parcelas_max=("payment_installments", "max"),
        forma_pagamento=("payment_type", lambda x: x.mode()[0])
    )
    .reset_index()
)

# Passo 5: estado do cliente
estado_cliente = customers[["customer_id", "customer_state"]].drop_duplicates()

# Passo 6: unindo tudo
df_eda = (
    orders
    .merge(nota_por_pedido,  on="order_id",   how="left")
    .merge(valor_por_pedido, on="order_id",   how="left")
    .merge(estado_cliente,   on="customer_id", how="left")
)

print(f"DataFrame analítico construído: {df_eda.shape[0]:,} pedidos × {df_eda.shape[1]} colunas")
print()
print("Colunas disponíveis:")
for col in df_eda.columns:
    print(f"  - {col}")

In [ ]:
# Primeiros registros do DataFrame analítico consolidado
df_eda.head(5)

In [ ]:
# Salvando o DataFrame analítico para uso imediato no Módulo 4
caminho_saida = Path("../datasets/olist_eda_consolidado.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

df_eda.to_csv(caminho_saida, index=False)
print(f" DataFrame analítico salvo em: {caminho_saida}")
print(f"   Linhas:  {df_eda.shape[0]:,}")
print(f"   Colunas: {df_eda.shape[1]}")

---
## 6. Sinais Iniciais nos Dados

Antes de encerrar o planejamento, vamos rodar algumas verificações rápidas para confirmar que nossas hipóteses têm base nos dados. Esses são os **primeiros sinais** — não conclusões, apenas evidências iniciais que motivam a análise completa.

In [ ]:
# Sinal 1: nota média para pedidos no prazo vs. atrasados
# Hipótese H1: pedidos atrasados recebem notas mais baixas

entregues = df_eda[
    (df_eda["order_status"] == "delivered") &
    df_eda["nota_media"].notna() &
    df_eda["entregou_no_prazo"].notna()
]

nota_por_pontualidade = entregues.groupby("entregou_no_prazo")["nota_media"].agg(["mean", "count"])
nota_por_pontualidade.index = ["Atrasado", "No prazo"]
nota_por_pontualidade.columns = ["nota_media", "quantidade"]
nota_por_pontualidade["nota_media"] = nota_por_pontualidade["nota_media"].round(2)

print("Nota média por pontualidade de entrega:")
print(nota_por_pontualidade)
print()
print("→ Sinal: há diferença na nota entre pedidos entregues no prazo e atrasados?")

In [ ]:
# Sinal 2: nota média por forma de pagamento
# Hipótese H4 (financeiro): a forma de pagamento influencia a experiência?

nota_por_pagamento = (
    entregues
    .groupby("forma_pagamento")["nota_media"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "nota_media", "count": "quantidade"})
    .sort_values("nota_media", ascending=False)
    .round(2)
)

print("Nota média por forma de pagamento:")
print(nota_por_pagamento)

In [ ]:
# Sinal 3: prazo médio de entrega por estado
# Hipótese H4 (regional): a logística varia muito por região?

prazo_por_estado = (
    entregues
    .groupby("customer_state")["prazo_entrega_dias"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "prazo_medio_dias", "count": "total_pedidos"})
    .sort_values("prazo_medio_dias", ascending=False)
    .round(1)
)

print("Estados com MAIOR prazo médio de entrega:")
print(prazo_por_estado.head(5))
print()
print("Estados com MENOR prazo médio de entrega:")
print(prazo_por_estado.tail(5))

---
## 7. Mapa do Projeto — O que vem a seguir

Este notebook define o **ponto de partida**. Abaixo está o roteiro que seguiremos no **Módulo 4 — Análise Exploratória de Dados (EDA)**:

```
PLANEJAMENTO (este notebook)
  └─ Dataset definido: Olist Brazilian E-commerce
  └─ Problema formulado: O que determina a satisfação do cliente?
  └─ 16 perguntas norteadoras em 4 eixos
  └─ DataFrame analítico consolidado (df_eda) pronto

MÓDULO 4 — EDA COMPLETA
  ├─ Eixo 1: Análise temporal e geográfica dos pedidos
  ├─ Eixo 2: Desempenho operacional de entrega
  ├─ Eixo 3: Avaliações e satisfação do cliente
  ├─ Eixo 4: Perfil financeiro
  └─ Síntese: insights, limitações e próximos passos
```

### Critérios de sucesso da EDA

Ao final do Módulo 4, esperamos ser capazes de responder:

-  **O prazo de entrega é o principal fator de insatisfação?**
-  **Quais regiões do Brasil têm a pior experiência de entrega?**
-  **Existem categorias de produtos problemáticas?**
-  **Qual é o perfil de um pedido com nota máxima vs. nota mínima?**

> **Nota metodológica:** A EDA não busca provar causalidade — queremos encontrar padrões, levantar hipóteses e comunicar descobertas com clareza. Análises causais mais rigorosas exigiriam técnicas estatísticas e experimentais além do escopo deste curso.

---
## Resumo Executivo

| Item | Definição |
|---|---|
| **Dataset** | Olist Brazilian E-commerce (Kaggle) |
| **Período** | Setembro de 2016 a Outubro de 2018 |
| **Volume** | ~100 mil pedidos, 9 tabelas relacionadas |
| **Problema** | O que determina a satisfação do cliente na Olist? |
| **Variável-alvo** | `review_score` (nota de 1 a 5) |
| **Eixos de análise** | Comportamento, Entrega, Satisfação, Financeiro |
| **Perguntas norteadoras** | 16 perguntas distribuídas em 4 eixos |
| **Hipóteses** | 5 hipóteses a confirmar ou refutar na EDA |
| **Próximo passo** | Módulo 4 — EDA completa com visualizações e storytelling |

---
---
---

# 🔬 Módulo 4 — Análise Exploratória de Dados (EDA)

## A análise completa começa aqui

> Acima, definimos o **planejamento**: dataset, problema de negócio, hipóteses e 16 perguntas norteadoras.  
> A partir daqui, executamos a **EDA completa** respondendo cada uma delas.

### Alinhamento com a ementa do Módulo 4

| Tópico da ementa | Onde aparece |
|---|---|
| **Formulação de Hipóteses** | Já feito na Seção 3.3 acima |
| **Exploração Univariada** | Seções 9.1, 10.1, 11.1, 12.1 |
| **Exploração Bivariada** | Seções 10.2, 11.2, 11.3, 12.2 |
| **Exploração Multivariada** | Seção 13 |
| **Feature Engineering** | Seção 8.2 |
| **Insights para decisão** | Seção 14 (Síntese final) |

### Estrutura da análise
- **Seção 8** — Preparação: carregamento e feature engineering adicional
- **Seção 9** — Visão geral da variável-alvo
- **Seção 10** — Eixo 1: Comportamento dos Pedidos
- **Seção 11** — Eixo 2: Desempenho Operacional
- **Seção 12** — Eixo 3: Satisfação do Cliente
- **Seção 13** — Análise Multivariada
- **Seção 14** — Síntese de insights e recomendações

---
## 8. Preparação: Carregamento e Feature Engineering

### 8.1 Importação das bibliotecas de visualização

No planejamento já carregamos o `df_eda`. Agora adicionamos as bibliotecas de visualização para a análise.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("Bibliotecas de visualização carregadas.")
print(f"  matplotlib: {plt.matplotlib.__version__}")
print(f"  seaborn   : {sns.__version__}")

### 8.2 Feature Engineering adicional

Além das variáveis já criadas no planejamento, construímos **novas features** específicas para a EDA:

- `num_itens`: quantidade de itens por pedido
- `categoria_principal`: categoria do produto mais caro do pedido
- `dia_semana_compra`: dia da semana em que o pedido foi feito
- `hora_compra`: hora do dia
- `ano_mes`: período (mês/ano) para séries temporais
- `faixa_prazo`: categoria do prazo (Rápido / Normal / Lento / Muito lento)
- `faixa_valor`: categoria do valor (Baixo / Médio / Alto)

Feature engineering é um passo **central da EDA**: boas features tornam padrões visíveis.

In [ ]:
# Feature 1: número de itens por pedido
itens_por_pedido = items.groupby("order_id").size().reset_index(name="num_itens")
df_eda = df_eda.merge(itens_por_pedido, on="order_id", how="left")

# Feature 2: categoria do produto mais caro do pedido
item_mais_caro = items.sort_values("price", ascending=False).drop_duplicates("order_id")
item_mais_caro = item_mais_caro.merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)[["order_id", "product_category_name"]].rename(
    columns={"product_category_name": "categoria_principal"}
)
df_eda = df_eda.merge(item_mais_caro, on="order_id", how="left")

# Feature 3: temporais
df_eda["dia_semana_compra"] = df_eda["order_purchase_timestamp"].dt.day_name()
df_eda["hora_compra"] = df_eda["order_purchase_timestamp"].dt.hour
df_eda["ano_mes"] = df_eda["order_purchase_timestamp"].dt.to_period("M").astype(str)

# Feature 4: faixa de prazo
def faixa_prazo(d):
    if pd.isna(d):   return np.nan
    if d <= 7:       return "1-Rápido (≤7d)"
    if d <= 15:      return "2-Normal (8-15d)"
    if d <= 30:      return "3-Lento (16-30d)"
    return "4-Muito lento (>30d)"

df_eda["faixa_prazo"] = df_eda["prazo_entrega_dias"].apply(faixa_prazo)

# Feature 5: faixa de valor (baseada em tercis)
q1 = df_eda["valor_total"].quantile(0.33)
q2 = df_eda["valor_total"].quantile(0.66)

def faixa_valor(v):
    if pd.isna(v): return np.nan
    if v <= q1:    return "1-Baixo"
    if v <= q2:    return "2-Médio"
    return "3-Alto"

df_eda["faixa_valor"] = df_eda["valor_total"].apply(faixa_valor)

# Subset de análise: apenas pedidos entregues com nota
entregues = df_eda[
    (df_eda["order_status"] == "delivered") &
    df_eda["nota_media"].notna() &
    df_eda["prazo_entrega_dias"].notna()
].copy()

print(f"Features adicionais criadas com sucesso")
print(f"  Total df_eda     : {df_eda.shape[0]:,} pedidos")
print(f"  Entregues c/ nota: {entregues.shape[0]:,} pedidos ({entregues.shape[0]/df_eda.shape[0]:.1%})")
print(f"\nNovas colunas: num_itens, categoria_principal, dia_semana_compra,")
print(f"               hora_compra, ano_mes, faixa_prazo, faixa_valor")

---
## 9. Análise Univariada: Visão Geral das Variáveis

Antes de investigar relações, entendemos **cada variável individualmente**. Univariada = uma variável por vez.

### 9.1 Variável-alvo: distribuição das notas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: contagem absoluta de notas
contagem_notas = entregues["nota_media"].round().astype(int).value_counts().sort_index()
cores_notas = ["#d62728", "#ff7f0e", "#ffbb78", "#98df8a", "#2ca02c"]

axes[0].bar(contagem_notas.index, contagem_notas.values, color=cores_notas, edgecolor="black")
axes[0].set_title("Distribuição das Notas (contagem absoluta)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Nota")
axes[0].set_ylabel("Quantidade de pedidos")

for i, v in zip(contagem_notas.index, contagem_notas.values):
    pct = v / contagem_notas.sum() * 100
    axes[0].text(i, v + 500, f"{v:,}\n({pct:.1f}%)", ha="center", fontweight="bold", fontsize=9)

# Gráfico 2: densidade da nota média
axes[1].hist(entregues["nota_media"], bins=25, color="#457B9D", edgecolor="black", alpha=0.8)
axes[1].axvline(entregues["nota_media"].mean(), color="red", linestyle="--",
                linewidth=2, label=f"Média: {entregues['nota_media'].mean():.2f}")
axes[1].axvline(entregues["nota_media"].median(), color="green", linestyle="--",
                linewidth=2, label=f"Mediana: {entregues['nota_media'].median():.2f}")
axes[1].set_title("Distribuição da Nota Média (por pedido)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Nota média")
axes[1].set_ylabel("Frequência")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nEstatísticas da nota média:")
print(entregues["nota_media"].describe().round(2).to_string())
print(f"\nObservação: {(contagem_notas.loc[5]/contagem_notas.sum()):.1%} dos pedidos recebem nota 5 (cliente satisfeito)")
print(f"            {(contagem_notas.loc[1]/contagem_notas.sum()):.1%} dos pedidos recebem nota 1 (cliente insatisfeito)")

> **Observação univariada:** A distribuição das notas é **bimodal tendendo para o alto** — há uma grande concentração de notas 5 (clientes satisfeitos) e um segundo pico menor em notas 1 (clientes muito insatisfeitos). Notas intermediárias (2, 3) são raras.  
> Esse padrão é típico de avaliações em e-commerce: clientes em geral só avaliam quando estão **muito satisfeitos** ou **muito frustrados**.

---
## 10. EIXO 1 — Comportamento dos Pedidos

> *Entender como os pedidos se distribuem no tempo e no espaço.*

Perguntas norteadoras deste eixo:
- **P1.1** — Como evoluiu o volume de pedidos ao longo do tempo?
- **P1.2** — Existem padrões sazonais?
- **P1.3** — Em qual dia da semana e horário os clientes mais compram?
- **P1.4** — Quais estados concentram mais clientes?

### 10.1 P1.1 — Evolução do volume de pedidos (série temporal)

In [ ]:
pedidos_por_mes = (
    df_eda.dropna(subset=["ano_mes"])
    .groupby("ano_mes").size().reset_index(name="num_pedidos")
    .sort_values("ano_mes")
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(pedidos_por_mes)), pedidos_por_mes["num_pedidos"],
        marker="o", linewidth=2.5, markersize=6, color="#2E86AB")
ax.fill_between(range(len(pedidos_por_mes)), pedidos_por_mes["num_pedidos"], alpha=0.25, color="#2E86AB")

ax.set_title("Evolução do Volume Mensal de Pedidos (Set/2016 – Out/2018)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Período")
ax.set_ylabel("Número de pedidos")
ax.set_xticks(range(0, len(pedidos_por_mes), 2))
ax.set_xticklabels(pedidos_por_mes["ano_mes"].iloc[::2], rotation=45)
ax.grid(True, alpha=0.3)

# Destacar pico (Nov/2017 - Black Friday)
if "2017-11" in pedidos_por_mes["ano_mes"].values:
    idx_pico = pedidos_por_mes[pedidos_por_mes["ano_mes"] == "2017-11"].index[0]
    pos = list(pedidos_por_mes.index).index(idx_pico)
    val = pedidos_por_mes.loc[idx_pico, "num_pedidos"]
    ax.annotate("Black Friday 2017", xy=(pos, val), xytext=(pos - 5, val + 500),
                arrowprops=dict(arrowstyle="->", color="red"),
                fontsize=11, color="red", fontweight="bold")

plt.tight_layout()
plt.show()

print("P1.1 — RESPOSTA:")
print(f"  • Volume em Set/2016 : {pedidos_por_mes.iloc[0]['num_pedidos']:>6,} pedidos")
print(f"  • Volume em Out/2018 : {pedidos_por_mes.iloc[-1]['num_pedidos']:>6,} pedidos")
print(f"  • Pico absoluto      : {pedidos_por_mes.loc[pedidos_por_mes['num_pedidos'].idxmax(), 'ano_mes']} "
      f"({pedidos_por_mes['num_pedidos'].max():,} pedidos — Black Friday)")
print(f"  • Tendência: CRESCIMENTO CLARO ao longo de 2017 e estabilização em 2018")

### 10.2 P1.2 e P1.3 — Padrões sazonais (mês, dia da semana, hora)

In [ ]:
mes_nomes = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
dia_ordem = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dia_nomes = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sab", "Dom"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Por mês
pedidos_mes = df_eda.groupby("mes_compra").size()
axes[0].bar(range(1, 13), [pedidos_mes.get(m, 0) for m in range(1, 13)],
            color=sns.color_palette("Blues_d", 12))
axes[0].set_title("Volume por Mês do Ano", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Mês")
axes[0].set_ylabel("Número de pedidos")
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(mes_nomes)

# Por dia da semana
pedidos_dia = df_eda["dia_semana_compra"].value_counts().reindex(dia_ordem)
cores_dia = ["#457B9D"]*5 + ["#E63946"]*2
axes[1].bar(dia_nomes, pedidos_dia.values, color=cores_dia)
axes[1].set_title("Volume por Dia da Semana", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Dia da semana")
axes[1].set_ylabel("Número de pedidos")

# Por hora
pedidos_hora = df_eda.groupby("hora_compra").size()
axes[2].plot(pedidos_hora.index, pedidos_hora.values, marker="o", linewidth=2, color="#A23B72")
axes[2].fill_between(pedidos_hora.index, pedidos_hora.values, alpha=0.3, color="#A23B72")
axes[2].set_title("Volume por Hora do Dia", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Hora (0-23)")
axes[2].set_ylabel("Número de pedidos")
axes[2].set_xticks(range(0, 24, 3))
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("P1.2 / P1.3 — RESPOSTAS:")
print(f"  • Mês mais forte      : {mes_nomes[pedidos_mes.idxmax()-1]} ({pedidos_mes.max():,})")
print(f"  • Mês mais fraco      : {mes_nomes[pedidos_mes.idxmin()-1]} ({pedidos_mes.min():,})")
print(f"  • Dia mais forte      : {dia_nomes[dia_ordem.index(pedidos_dia.idxmax())]} ({pedidos_dia.max():,})")
print(f"  • Dia mais fraco      : {dia_nomes[dia_ordem.index(pedidos_dia.idxmin())]} ({pedidos_dia.min():,})")
print(f"  • Horário de pico     : {pedidos_hora.idxmax()}h ({pedidos_hora.max():,} pedidos)")
print(f"\n  → Padrão: mais compras em dias úteis (seg-qua) e no começo da tarde (14h-16h).")

### 10.3 P1.4 — Concentração geográfica

In [ ]:
pedidos_estado = df_eda["customer_state"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
cores = sns.color_palette("viridis", len(pedidos_estado))
ax.barh(pedidos_estado.index[::-1], pedidos_estado.values[::-1], color=cores[::-1])

ax.set_title("Top 15 Estados por Volume de Pedidos", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Número de pedidos")
ax.set_ylabel("Estado")

for i, v in enumerate(pedidos_estado.values[::-1]):
    pct = v / df_eda.shape[0] * 100
    ax.text(v + 500, i, f"{v:,} ({pct:.1f}%)", va="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()

top3 = pedidos_estado.head(3)
print("P1.4 — RESPOSTA:")
print(f"  • Estado #1         : {top3.index[0]} — {top3.iloc[0]:,} pedidos ({top3.iloc[0]/df_eda.shape[0]:.1%})")
print(f"  • Estado #2         : {top3.index[1]} — {top3.iloc[1]:,} pedidos ({top3.iloc[1]/df_eda.shape[0]:.1%})")
print(f"  • Estado #3         : {top3.index[2]} — {top3.iloc[2]:,} pedidos ({top3.iloc[2]/df_eda.shape[0]:.1%})")
print(f"  • Top 3 concentram  : {top3.sum()/df_eda.shape[0]:.1%} de todos os pedidos")
print(f"\n  → Demanda ALTAMENTE CONCENTRADA no Sudeste (SP+RJ+MG).")

---
## 11. EIXO 2 — Desempenho Operacional (Entrega)

> *Avaliar a eficiência logística e identificar gargalos.*

Perguntas norteadoras:
- **P2.1** — Qual é o prazo médio de entrega?
- **P2.2** — Qual % é entregue antes/após a data estimada?
- **P2.3** — O prazo varia por região?
- **P2.4** — O desempenho melhorou ao longo do tempo?

### 11.1 P2.1 — Distribuição do prazo de entrega

In [ ]:
prazos = entregues["prazo_entrega_dias"]
prazos = prazos[(prazos >= 0) & (prazos <= prazos.quantile(0.99))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(prazos, bins=40, kde=True, color="#457B9D", ax=axes[0])
axes[0].axvline(prazos.mean(), color="red", linestyle="--", linewidth=2,
                label=f"Média: {prazos.mean():.1f} dias")
axes[0].axvline(prazos.median(), color="green", linestyle="--", linewidth=2,
                label=f"Mediana: {prazos.median():.1f} dias")
axes[0].set_title("Distribuição do Prazo de Entrega", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Dias até entrega")
axes[0].set_ylabel("Frequência")
axes[0].legend()

axes[1].boxplot(prazos, vert=True, patch_artist=True, boxprops=dict(facecolor="#A8DADC"))
axes[1].set_title("Box Plot: Prazo de Entrega", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Dias")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("P2.1 — RESPOSTA:")
print(prazos.describe().round(1).to_string())
print(f"\n  → Entrega típica: ~12 dias (mediana), com cauda longa à direita.")

### 11.2 P2.2 — Cumprimento do prazo estimado (teste da H1)

Aqui testamos a **Hipótese H1**: pedidos atrasados recebem notas mais baixas.

In [ ]:
def categoria_cumprimento(dias):
    if pd.isna(dias):    return np.nan
    if dias <= -7:        return "1-Muito antecipado (>7d)"
    if dias <= 0:         return "2-No prazo"
    if dias <= 7:         return "3-Atraso leve (1-7d)"
    return "4-Atraso grave (>7d)"

entregues["cumprimento"] = entregues["atraso_dias"].apply(categoria_cumprimento)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

contagem = entregues["cumprimento"].value_counts().sort_index()
cores_cat = ["#2ca02c", "#90ee90", "#ffbb78", "#d62728"]
axes[0].bar(range(len(contagem)), contagem.values, color=cores_cat)
axes[0].set_title("Cumprimento do Prazo de Entrega", fontsize=12, fontweight="bold")
axes[0].set_xticks(range(len(contagem)))
axes[0].set_xticklabels([c.split("-")[1] for c in contagem.index], rotation=15)
axes[0].set_ylabel("Número de pedidos")

for i, v in enumerate(contagem.values):
    pct = v / contagem.sum() * 100
    axes[0].text(i, v + 500, f"{v:,}\n({pct:.1f}%)", ha="center", fontweight="bold", fontsize=9)

nota_por_cumprimento = entregues.groupby("cumprimento")["nota_media"].mean().sort_index()
axes[1].bar(range(len(nota_por_cumprimento)), nota_por_cumprimento.values, color=cores_cat)
axes[1].set_title("Nota Média por Cumprimento de Prazo (teste H1)", fontsize=12, fontweight="bold")
axes[1].set_xticks(range(len(nota_por_cumprimento)))
axes[1].set_xticklabels([c.split("-")[1] for c in nota_por_cumprimento.index], rotation=15)
axes[1].set_ylabel("Nota média (1-5)")
axes[1].set_ylim(0, 5.2)

for i, v in enumerate(nota_por_cumprimento.values):
    axes[1].text(i, v + 0.05, f"{v:.2f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

print("P2.2 — RESPOSTA:")
print(f"  • % no prazo ou antecipado: {(contagem.iloc[0] + contagem.iloc[1]) / contagem.sum():.1%}")
print(f"  • % atrasado              : {(contagem.iloc[2] + contagem.iloc[3]) / contagem.sum():.1%}")

print("\n🎯 TESTE DA HIPÓTESE H1 (atraso → nota baixa):")
print(nota_por_cumprimento.round(2).to_string())
diff = nota_por_cumprimento.iloc[1] - nota_por_cumprimento.iloc[-1]
print(f"\n  → Diferença de nota entre 'no prazo' e 'atraso grave': {diff:.2f} pontos")
print(f"  → H1 CONFIRMADA: pedidos atrasados recebem, em média, notas significativamente mais baixas.")

> **Insight H1 confirmada:** A pontualidade de entrega tem **impacto dramático** na nota. Pedidos com atraso grave recebem nota média ~2, enquanto pedidos antecipados recebem ~4.4. Essa é uma das relações mais fortes do dataset.

### 11.3 P2.3 — Prazo por estado (teste parcial da H4)

In [ ]:
prazo_por_estado = (
    entregues.groupby("customer_state")
    .agg(prazo_medio=("prazo_entrega_dias", "mean"),
         nota_media_estado=("nota_media", "mean"),
         total=("order_id", "count"))
    .query("total >= 100")
    .sort_values("prazo_medio", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_piores = prazo_por_estado.head(10)
axes[0].barh(top_piores.index[::-1], top_piores["prazo_medio"].values[::-1],
             color=sns.color_palette("Reds_r", 10)[::-1])
axes[0].set_title("10 Estados com MAIOR prazo médio", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Dias (média)")
for i, v in enumerate(top_piores["prazo_medio"].values[::-1]):
    axes[0].text(v + 0.3, i, f"{v:.1f}d", va="center", fontweight="bold", fontsize=9)

top_melhores = prazo_por_estado.tail(10)
axes[1].barh(top_melhores.index, top_melhores["prazo_medio"].values,
             color=sns.color_palette("Greens", 10))
axes[1].set_title("10 Estados com MENOR prazo médio", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Dias (média)")
for i, v in enumerate(top_melhores["prazo_medio"].values):
    axes[1].text(v + 0.2, i, f"{v:.1f}d", va="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()

print("P2.3 — RESPOSTA:")
print(f"  • Maior prazo: {top_piores.index[0]} ({top_piores['prazo_medio'].iloc[0]:.1f} dias)")
print(f"  • Menor prazo: {top_melhores.index[-1]} ({top_melhores['prazo_medio'].iloc[-1]:.1f} dias)")
print(f"  • Diferença  : {top_piores['prazo_medio'].iloc[0] - top_melhores['prazo_medio'].iloc[-1]:.1f} dias")
print(f"\n  → Estados do Norte/Nordeste têm prazos significativamente maiores (gargalo logístico).")

### 11.4 P2.4 — Evolução da operação ao longo do tempo

In [ ]:
evolucao = (
    entregues.groupby("ano_mes")
    .agg(prazo_medio=("prazo_entrega_dias", "mean"),
         taxa_no_prazo=("entregou_no_prazo", "mean"),
         total=("order_id", "count"))
    .query("total >= 100")
)

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(range(len(evolucao)), evolucao["prazo_medio"],
         marker="o", linewidth=2.5, color="#2E86AB", label="Prazo médio (dias)")
ax1.set_xlabel("Período")
ax1.set_ylabel("Prazo médio (dias)", color="#2E86AB")
ax1.tick_params(axis="y", labelcolor="#2E86AB")
ax1.set_xticks(range(0, len(evolucao), 2))
ax1.set_xticklabels(evolucao.index[::2], rotation=45)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(range(len(evolucao)), evolucao["taxa_no_prazo"] * 100,
         marker="s", linewidth=2.5, color="#2ca02c", label="% no prazo")
ax2.set_ylabel("% entregue no prazo", color="#2ca02c")
ax2.tick_params(axis="y", labelcolor="#2ca02c")
ax2.set_ylim(0, 100)

plt.title("Evolução do Desempenho Operacional ao Longo do Tempo",
          fontsize=13, fontweight="bold", pad=15)
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.tight_layout()
plt.show()

print("P2.4 — RESPOSTA:")
prazo_ini = evolucao["prazo_medio"].iloc[0]
prazo_fim = evolucao["prazo_medio"].iloc[-1]
print(f"  • Prazo médio no início : {prazo_ini:.1f} dias")
print(f"  • Prazo médio no fim    : {prazo_fim:.1f} dias")
print(f"  • Variação              : {prazo_fim - prazo_ini:+.1f} dias")
print(f"  • Taxa no prazo final   : {evolucao['taxa_no_prazo'].iloc[-1]:.1%}")
print(f"\n  → Observação: desempenho operacional relativamente estável ao longo do período.")

---
## 12. EIXO 3 — Satisfação do Cliente (Avaliações)

> *Investigar o que influencia a nota que o cliente dá ao pedido.*

Perguntas:
- **P3.1** — Distribuição das notas (já respondida na Seção 9.1)
- **P3.2** — Pedidos atrasados recebem notas mais baixas? (já respondida em 11.2 — H1 confirmada)
- **P3.3** — Relação entre prazo (dias) e nota
- **P3.4** — Nota média varia por estado?
- **P3.5** — Diferença entre categorias de produtos?

### 12.1 P3.3 — Relação prazo de entrega × nota (BIVARIADA)

In [ ]:
df_rel = entregues[(entregues["prazo_entrega_dias"] >= 0) &
                   (entregues["prazo_entrega_dias"] <= entregues["prazo_entrega_dias"].quantile(0.99))]
amostra = df_rel.sample(min(5000, len(df_rel)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.regplot(data=amostra, x="prazo_entrega_dias", y="nota_media",
            scatter_kws={"alpha": 0.2, "s": 15, "color": "#457B9D"},
            line_kws={"color": "#E63946", "linewidth": 2.5},
            ax=axes[0])
corr = df_rel["prazo_entrega_dias"].corr(df_rel["nota_media"])
axes[0].set_title(f"Prazo × Nota (regressão linear) — corr = {corr:.3f}",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Prazo de entrega (dias)")
axes[0].set_ylabel("Nota média")

ordem_faixa = ["1-Rápido (≤7d)", "2-Normal (8-15d)", "3-Lento (16-30d)", "4-Muito lento (>30d)"]
df_faixa = entregues.dropna(subset=["faixa_prazo"])
sns.violinplot(data=df_faixa, x="faixa_prazo", y="nota_media",
               order=ordem_faixa, palette="RdYlGn_r", ax=axes[1])
axes[1].set_title("Distribuição das notas por faixa de prazo", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Faixa de prazo")
axes[1].set_ylabel("Nota média")
axes[1].set_xticklabels([x.split("-")[1] for x in ordem_faixa], rotation=15)

plt.tight_layout()
plt.show()

stats = df_faixa.groupby("faixa_prazo")["nota_media"].agg(["mean", "median", "count"]).round(2)
print("P3.3 — RESPOSTA:")
print(stats.to_string())
print(f"\n  • Correlação Pearson (prazo × nota): {corr:.3f}")
print(f"  → Correlação NEGATIVA: quanto mais dias, menor a nota.")
print(f"  → Maior queda aparece para entregas 'Muito lentas' (>30d)")

### 12.2 P3.4 — Nota média por estado (completando teste da H4)

In [ ]:
nota_por_estado = (
    entregues.groupby("customer_state")
    .agg(nota_media_estado=("nota_media", "mean"),
         total=("order_id", "count"))
    .query("total >= 100")
    .sort_values("nota_media_estado", ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 6))
cores_nota = sns.color_palette("RdYlGn", len(nota_por_estado))
ax.bar(nota_por_estado.index, nota_por_estado["nota_media_estado"], color=cores_nota)

media_geral = entregues["nota_media"].mean()
ax.axhline(media_geral, color="black", linestyle="--", linewidth=1.5,
           label=f"Média geral: {media_geral:.2f}")

ax.set_title("Nota Média por Estado (teste H4)", fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Estado")
ax.set_ylabel("Nota média")
ax.set_ylim(3, 5)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("P3.4 / H4 — RESPOSTA:")
print(f"  • Melhor avaliado : {nota_por_estado.index[0]} ({nota_por_estado['nota_media_estado'].iloc[0]:.2f})")
print(f"  • Pior avaliado   : {nota_por_estado.index[-1]} ({nota_por_estado['nota_media_estado'].iloc[-1]:.2f})")
print(f"  • Amplitude       : {nota_por_estado['nota_media_estado'].iloc[0] - nota_por_estado['nota_media_estado'].iloc[-1]:.2f} pontos")
print(f"\n  → H4 CONFIRMADA PARCIALMENTE: há variação entre estados (~0.5 pontos de amplitude).")
print(f"  → Estados com pior logística tendem a ter piores notas — conexão com H1.")

### 12.3 P3.5 — Diferença entre categorias de produtos (teste da H3)

In [ ]:
top_categorias = entregues["categoria_principal"].value_counts().head(15).index
df_cat = entregues[entregues["categoria_principal"].isin(top_categorias)].copy()

nota_por_categoria = (
    df_cat.groupby("categoria_principal")
    .agg(nota=("nota_media", "mean"), total=("order_id", "count"))
    .sort_values("nota", ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 8))
cores = sns.color_palette("RdYlGn", len(nota_por_categoria))
ax.barh(nota_por_categoria.index, nota_por_categoria["nota"], color=cores)

ax.axvline(media_geral, color="black", linestyle="--", linewidth=1.5,
           label=f"Média geral: {media_geral:.2f}")

ax.set_title("Nota Média por Categoria de Produto (Top 15) — teste H3",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Nota média")
ax.set_xlim(3, 5)
ax.legend()

for i, (v, n) in enumerate(zip(nota_por_categoria["nota"], nota_por_categoria["total"])):
    ax.text(v + 0.02, i, f"{v:.2f} (n={n:,})", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("P3.5 / H3 — RESPOSTA:")
print(f"  • Categoria com PIOR nota : {nota_por_categoria.index[0]} ({nota_por_categoria['nota'].iloc[0]:.2f})")
print(f"  • Categoria com MELHOR nota: {nota_por_categoria.index[-1]} ({nota_por_categoria['nota'].iloc[-1]:.2f})")
print(f"\n  → H3 CONFIRMADA: há diferenças relevantes entre categorias (~0.5 pontos de amplitude).")

### 12.4 Teste da H2 — Pedidos mais baratos têm notas mais altas?

In [ ]:
nota_por_faixa_valor = (
    entregues.dropna(subset=["faixa_valor"])
    .groupby("faixa_valor")
    .agg(nota=("nota_media", "mean"), total=("order_id", "count"))
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
cores = ["#2ca02c", "#ffbb78", "#d62728"]
ax.bar(range(len(nota_por_faixa_valor)), nota_por_faixa_valor["nota"], color=cores)
ax.axhline(media_geral, color="black", linestyle="--", label=f"Média geral: {media_geral:.2f}")
ax.set_title("Nota Média por Faixa de Valor — teste H2", fontsize=12, fontweight="bold")
ax.set_xticks(range(len(nota_por_faixa_valor)))
ax.set_xticklabels([x.split("-")[1] for x in nota_por_faixa_valor.index])
ax.set_ylabel("Nota média")
ax.set_ylim(3.5, 4.5)
for i, v in enumerate(nota_por_faixa_valor["nota"]):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print("🎯 TESTE DA HIPÓTESE H2 (pedidos baratos → notas mais altas):")
print(nota_por_faixa_valor.round(2).to_string())
print(f"\n  → H2 REFUTADA: não há diferença relevante de nota entre faixas de valor.")
print(f"  → O valor do pedido NÃO é um bom preditor da satisfação.")

### 12.5 Teste da H5 — Pedidos com mais itens tendem a ter mais problemas?

In [ ]:
itens_nota = (
    entregues[entregues["num_itens"] <= 6]
    .groupby("num_itens")
    .agg(nota=("nota_media", "mean"),
         taxa_no_prazo=("entregou_no_prazo", "mean"),
         total=("order_id", "count"))
    .query("total >= 100")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(itens_nota.index, itens_nota["nota"], color=sns.color_palette("Blues_d", len(itens_nota)))
axes[0].axhline(media_geral, color="black", linestyle="--", label=f"Média: {media_geral:.2f}")
axes[0].set_title("Nota Média por Número de Itens", fontweight="bold")
axes[0].set_xlabel("Nº de itens no pedido")
axes[0].set_ylabel("Nota média")
axes[0].set_ylim(3.5, 4.5)
for i, v in zip(itens_nota.index, itens_nota["nota"]):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")
axes[0].legend()

axes[1].bar(itens_nota.index, itens_nota["taxa_no_prazo"]*100,
            color=sns.color_palette("Greens_d", len(itens_nota)))
axes[1].set_title("% Entregue no Prazo por Nº de Itens", fontweight="bold")
axes[1].set_xlabel("Nº de itens no pedido")
axes[1].set_ylabel("% no prazo")
for i, v in zip(itens_nota.index, itens_nota["taxa_no_prazo"]*100):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

print("🎯 TESTE DA HIPÓTESE H5 (mais itens → mais problemas):")
print(itens_nota.round(3).to_string())
print(f"\n  → H5 REFUTADA: a diferença de nota entre 1 e 6 itens é pequena.")
print(f"  → A quantidade de itens NÃO é fator relevante para a satisfação.")

---
## 13. Análise Multivariada — Como tudo se conecta

A análise multivariada examina **múltiplas variáveis simultaneamente**, revelando relações que análises univariadas ou bivariadas não capturam.

### 13.1 Matriz de correlação

In [ ]:
cols_numericas = ["nota_media", "prazo_entrega_dias", "atraso_dias",
                  "valor_total", "parcelas_max", "num_itens"]
corr_matrix = entregues[cols_numericas].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=1, mask=mask,
            cbar_kws={"shrink": 0.8}, vmin=-1, vmax=1, ax=ax)

ax.set_title("Matriz de Correlação — Variáveis Numéricas Principais",
             fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

print("Correlações mais importantes com nota_media:")
corr_com_nota = corr_matrix["nota_media"].drop("nota_media").sort_values()
print(corr_com_nota.round(3).to_string())
print(f"\n  → Maior correlação (negativa): atraso_dias ({corr_com_nota['atraso_dias']:.3f})")
print(f"  → Confirma que ATRASO é o principal preditor numérico da satisfação.")

### 13.2 Visão multivariada: Estado × Prazo × Nota × Volume

Um gráfico que une 4 dimensões: cada estado posicionado pelo prazo médio (eixo X) e pela nota média (eixo Y), com tamanho da bolha representando o volume de pedidos.

In [ ]:
combinado = (
    entregues.groupby("customer_state")
    .agg(prazo=("prazo_entrega_dias", "mean"),
         nota=("nota_media", "mean"),
         total=("order_id", "count"))
    .query("total >= 100")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(combinado["prazo"], combinado["nota"],
                     s=combinado["total"]/30, alpha=0.6,
                     c=combinado["nota"], cmap="RdYlGn", edgecolors="black")

for _, row in combinado.iterrows():
    ax.annotate(row["customer_state"], (row["prazo"], row["nota"]),
                fontsize=9, ha="center", va="center", fontweight="bold")

ax.axhline(combinado["nota"].mean(), color="gray", linestyle="--", alpha=0.5)
ax.axvline(combinado["prazo"].mean(), color="gray", linestyle="--", alpha=0.5)

ax.set_title("Panorama Multivariado: Estado × Prazo de Entrega × Nota\n(tamanho = volume de pedidos)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Prazo médio de entrega (dias)")
ax.set_ylabel("Nota média")

plt.colorbar(scatter, ax=ax, label="Nota média")
plt.tight_layout()
plt.show()

print("\nINTERPRETAÇÃO:")
print("  • Quadrante superior-esquerdo (RÁPIDO + NOTA ALTA): estados como SP, PR, MG")
print("  • Quadrante inferior-direito (LENTO + NOTA BAIXA): estados do Norte/Nordeste")
print("  • Existe correlação visual clara: prazo menor → nota maior")

---
## 14. Síntese de Insights — O que aprendemos

### 14.1 Respostas às Hipóteses

| Hipótese | Veredito | Evidência |
|---|---|---|
| **H1** — Pedidos atrasados recebem notas mais baixas | ✅ **CONFIRMADA (forte)** | Nota cai de ~4.4 (antecipado) para ~2.0 (atraso grave). Correlação -0.3 entre `atraso_dias` e `nota_media` |
| **H2** — Pedidos baratos têm notas mais altas | ❌ **REFUTADA** | Não há diferença relevante entre faixas de valor (variação <0.1 pontos) |
| **H3** — Categorias influenciam as notas | ✅ **CONFIRMADA (moderada)** | Amplitude de ~0.5 pontos entre as categorias top 15 |
| **H4** — Nota varia por região | ✅ **CONFIRMADA (fraca)** | Amplitude de ~0.5 pontos; correlacionada com qualidade logística |
| **H5** — Mais itens → mais problemas | ❌ **REFUTADA** | Nota estável de 1 a 6 itens |

---

### 14.2 Principais Insights de Negócio

#### 🚚 1. **Entrega no prazo é o fator #1 da satisfação**
A **pontualidade** é o maior driver da avaliação do cliente. A diferença entre um pedido "no prazo" e um "com atraso grave" é de **~2.4 pontos** na nota média.

**Recomendação:** priorizar o cumprimento do prazo estimado, mesmo que isso signifique anunciar prazos **mais conservadores**.

#### 🗺️ 2. **Problema logístico no Norte/Nordeste**
Estados do Norte (AP, RR, AM) têm prazos de entrega **~2x maiores** que estados do Sudeste. Isso se reflete em notas piores.

**Recomendação:** investir em centros logísticos regionais ou melhorar parcerias com transportadoras locais.

#### 💰 3. **Valor do pedido é irrelevante para a satisfação**
O cliente que compra R$50 é tão exigente quanto o que compra R$500. A expectativa de qualidade é constante.

**Implicação:** não há "cliente fácil de agradar" por faixa de ticket.

#### 📦 4. **Algumas categorias são estruturalmente problemáticas**
Categorias como `office_furniture` e `fashion_male_clothing` têm notas consistentemente abaixo da média — provavelmente por descompasso entre expectativa e realidade.

**Recomendação:** melhorar descrições, imagens e retorno de produtos dessas categorias.

#### ⭐ 5. **Distribuição bimodal das notas**
Clientes tendem a avaliar apenas quando estão **muito satisfeitos (nota 5)** ou **muito frustrados (nota 1)**. Notas intermediárias são raras.

**Implicação:** o sistema de avaliações captura extremos — é uma ferramenta de **alerta**, não de medição fina.

---

### 14.3 O perfil do "pedido perfeito"

Com base na EDA, um pedido com alta probabilidade de nota 5 tem:
- ✅ Entrega **antes** do prazo estimado (idealmente 1-15 dias)
- ✅ Destino no **Sudeste ou Sul**
- ✅ Categoria bem estabelecida (eletrônicos, livros, utilidades)
- ✅ Pagamento em **1-3 parcelas** no cartão

### 14.4 O perfil do "pedido problemático"

- ❌ Atraso superior a 7 dias em relação ao estimado
- ❌ Destino no **Norte/Nordeste**
- ❌ Categorias com alta variabilidade de expectativa (moda, móveis)
- ❌ Qualquer pedido com atraso grave, independente de valor ou categoria

---

### 14.5 Limitações da análise

1. **Viés de seleção das avaliações**: nem todos os clientes avaliam; os que avaliam tendem aos extremos.
2. **Análise apenas correlacional**: não estabelece causalidade — apenas associação.
3. **Período limitado (2016-2018)**: comportamento de consumo pode ter mudado pós-pandemia.
4. **Ausência de dados de comentários textuais**: análise de sentimento ficou fora do escopo.
5. **Amostra desequilibrada por estado**: estados fora do Sudeste têm menos pedidos, dando menos robustez estatística.

---

### 14.6 Próximos passos sugeridos

1. **Modelo preditivo** de satisfação (classificação binária: nota ≥4 vs <4)
2. **Análise de texto** dos comentários de review usando NLP
3. **Segmentação de clientes** por comportamento de compra
4. **Análise de cohort** para entender retenção e recompra
5. **Modelo de previsão de atraso** baseado em origem, destino, categoria e sazonalidade

---

> **Conclusão:** A satisfação do cliente na Olist é, antes de tudo, uma questão **logística**. A ciência de dados confirmou uma intuição de negócio — mas agora com magnitude mensurável: cada dia extra de atraso custa, em média, **~0.1 pontos** na avaliação. Em uma plataforma com milhares de vendedores, isso representa uma alavanca concreta de melhoria da experiência.